# 实验 04 · 留出学习空间，看清训练带来的变化

上一课的初始模型在参考运行中已有 97.5% 的训练准确率，进一步改善的空间较小。为了观察学习过程，本课重新创建相同结构的微型 Qwen2，只做 200 步监督预热，再复制成三个参数完全相同的模型。

1. **冻结基线**：不再训练。
2. **继续 SFT**：直接使用标准答案，训练 1200 次参数更新。
3. **奖励微调 RL**：每题采样 8 个答案，训练 600 轮，每轮 2 次参数更新。

本课比较两件事：继续训练能否超过冻结基线，以及奖励微调与继续 SFT 各自表现如何。结果只适用于这个重新预热的起点，RL 是否更好要看实际评估。 数据划分沿用第二课；三个模型面对完全相同的 80 道测试题。固定训练步数，不选最优测试检查点。

这是单随机种子的教学演示，不是统计显著性结论。两种训练有相同的更新次数和每次 128 条回答序列，但采样成本、题目多样性、监督信息量不同，不能称为等算力比较。

选择 Python (verl GPU)，从头 Shift+Enter。无需模型下载，结果写入独立的 `experiment04_output/`，不覆盖前三课。

## 1 · 固定数据和随机种子
沿用第二课 320 道训练题、80 道测试题；交换加数的题目仍在同一分组。

In [ ]:
import sys, time, random, json
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import Qwen2Config, Qwen2ForCausalLM

assert sys.executable.startswith('/opt/verl-venv/'), '请选择 Python (verl GPU)'
assert torch.cuda.is_available(), '需要 CUDA GPU'
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)
device = 'cuda'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
import copy
vocab = '_0123456789+=#'
stoi = {c: i for i, c in enumerate(vocab)}
PAD, EOS = stoi['_'], stoi['#']
MAX_LEN = 10
def encode(text):
    return [stoi[c] for c in text]

groups = [(a, b) for a in range(20) for b in range(a, 20)]
random.Random(SEED).shuffle(groups)
heldout = set(groups[:42])
train_pairs, test_pairs = [], []
for a in range(20):
    for b in range(20):
        (test_pairs if tuple(sorted((a, b))) in heldout else train_pairs).append((a, b))
assert not {tuple(sorted(p)) for p in train_pairs} & {tuple(sorted(p)) for p in test_pairs}

def pack(pairs):
    ids = torch.full((len(pairs), MAX_LEN), PAD, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    mask = torch.zeros_like(ids)
    for i, (a, b) in enumerate(pairs):
        prompt = f'{a}+{b}='
        sequence = encode(prompt + str(a+b) + '#')
        ids[i, :len(sequence)] = torch.tensor(sequence)
        labels[i, len(prompt):len(sequence)] = ids[i, len(prompt):len(sequence)]
        mask[i, :len(sequence)] = 1
    return ids.to(device), labels.to(device), mask.to(device)

train_ids, train_labels, train_mask = pack(train_pairs)
print('训练题数:', len(train_pairs), '| 独立测试题数:', len(test_pairs))
print('字符:', list('12+7=19#'), '\ntoken:', encode('12+7=19#'))
print('示例 labels:', train_labels[0].tolist())

## 2 · 只预热 200 步，建立相同起点
这里只用训练题。初始化后先记下未训练模型表现；预热步数预先固定，不按测试准确率挑起点。

In [ ]:
config = Qwen2Config(vocab_size=len(vocab), hidden_size=128, intermediate_size=256,
    num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
    max_position_embeddings=32, pad_token_id=PAD, eos_token_id=EOS,
    tie_word_embeddings=True, attention_dropout=0.0)
model = Qwen2ForCausalLM(config).to(device)
model.config.use_cache = False
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')

@torch.no_grad()
def predict(pairs):
    model.eval()
    results = [''] * len(pairs)
    by_length = {}
    for i, (a, b) in enumerate(pairs):
        prompt = f'{a}+{b}='
        by_length.setdefault(len(prompt), []).append((i, prompt))
    for length, rows in by_length.items():
        x = torch.tensor([encode(p) for _, p in rows], device=device)
        for _ in range(3):
            nxt = model(input_ids=x).logits[:, -1].argmax(-1, keepdim=True)
            x = torch.cat((x, nxt), dim=1)
        for (i, _), tokens in zip(rows, x[:, length:].tolist()):
            results[i] = ''.join(vocab[t] for t in tokens).split('#')[0]
    return results

def accuracy(pairs):
    answers = predict(pairs)
    return sum(s == str(a+b) for s, (a,b) in zip(answers, pairs)) / len(pairs)

before_train = accuracy(train_pairs)
before_test = accuracy(test_pairs)
print(f'训练前：训练集 {before_train:.1%} / 测试集 {before_test:.1%}')
WARMUP = 200
warm_optimizer = torch.optim.AdamW(model.parameters(),lr=0.002,weight_decay=0.01)
for step in range(WARMUP):
    model.train()
    idx=torch.randint(len(train_pairs),(64,),device='cuda')
    loss=model(input_ids=train_ids[idx],attention_mask=train_mask[idx],labels=train_labels[idx]).loss
    warm_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    warm_optimizer.step()
model.eval()
actor=copy.deepcopy(model).eval()
reference=copy.deepcopy(model).eval().requires_grad_(False)
sft_model=copy.deepcopy(model).eval()
initial_state={k:v.detach().cpu().clone() for k,v in reference.state_dict().items()}
assert all(torch.equal(actor.state_dict()[k],sft_model.state_dict()[k]) for k in initial_state)
print('预热结束，三个分支参数完全相同。')
del warm_optimizer


## 3 · 建立奖励与基线
与第三课相同：只给采样得到的完整正确答案奖励 1，错误为 0。没有答案前缀泄漏，也不在 rollout 中提供标准答案。

In [ ]:
def encode(text):
    return [stoi[c] for c in text]

def decode(tokens):
    return ''.join(vocab[t] for t in tokens).split('#')[0]

@torch.no_grad()
def rollout(model, pairs, group=1, sample=True):
    prompts = [encode(f'{a}+{b}=') for a,b in pairs]
    assert len({len(p) for p in prompts}) == 1
    x = torch.tensor(prompts, device='cuda').repeat_interleave(group, dim=0)
    prompt_len = x.shape[1]
    active = torch.ones(x.shape[0], dtype=torch.bool, device='cuda')
    masks, old_logps = [], []
    for _ in range(3):
        logits = model(input_ids=x).logits[:, -1]
        distribution = torch.distributions.Categorical(logits=logits)
        nxt = distribution.sample() if sample else logits.argmax(-1)
        masks.append(active.clone())
        old_logps.append(distribution.log_prob(nxt))
        active = active & (nxt != EOS)
        x = torch.cat([x,nxt[:,None]],dim=1)
    answers = [decode(row) for row in x[:,prompt_len:].tolist()]
    targets = [str(a+b) for a,b in pairs for _ in range(group)]
    rewards = torch.tensor([float(a==b) for a,b in zip(answers,targets)],device='cuda')
    return x, torch.stack(masks,1), torch.stack(old_logps,1), rewards, answers, prompt_len

by_length = {}
for p in train_pairs:
    by_length.setdefault(len(f'{p[0]}+{p[1]}='), []).append(p)

def evaluate(model, pairs):
    buckets = {}
    answers = ['']*len(pairs)
    for i,p in enumerate(pairs):
        buckets.setdefault(len(f'{p[0]}+{p[1]}='), []).append((i,p))
    for rows in buckets.values():
        result = rollout(model,[p for _,p in rows],sample=False)[4]
        for (i,_),answer in zip(rows,result):
            answers[i] = answer
    acc = sum(s==str(a+b) for s,(a,b) in zip(answers,pairs))/len(pairs)
    return acc, answers

demo_pairs = by_length[min(by_length)][:2]
demo = rollout(actor,demo_pairs,group=8)
for i,p in enumerate(demo_pairs):
    print(f'{p[0]}+{p[1]}=：', list(zip(demo[4][i*8:(i+1)*8],demo[3][i*8:(i+1)*8].tolist())))
baseline_train,_ = evaluate(reference,train_pairs)
baseline_test,baseline_answers = evaluate(reference,test_pairs)
print(f'冻结基线：训练 {baseline_train:.2%} / 测试 {baseline_test:.2%}')
def group_advantages(rewards):
    return (rewards-rewards.mean(-1,keepdim=True))/(rewards.std(-1,keepdim=True,unbiased=False)+1e-6)

toy = torch.tensor([[1.,0.,1.,0.],[1.,1.,1.,1.]])
toy_adv = group_advantages(toy)
assert torch.allclose(toy_adv.mean(-1), torch.zeros(2), atol=1e-6)
assert torch.all(toy_adv[1]==0)
assert toy_adv[0,0]>0 and toy_adv[0,1]<0
print('优势示例:',toy_adv.tolist())
assert demo[1][:,0].all()
generated = demo[0][:,demo[5]:]
for i in range(1,3):
    assert torch.equal(demo[1][:,i], (generated[:,:i]!=EOS).all(1))
print('组内优势与结束符掩码检查通过。')

def response_logps(model,x,prompt_len):
    logits = model(input_ids=x).logits[:,prompt_len-1:-1]
    return F.log_softmax(logits.float(),dim=-1).gather(-1,x[:,prompt_len:,None]).squeeze(-1)

check = response_logps(actor,demo[0],demo[5]).detach()
torch.testing.assert_close(check,demo[2],atol=2e-5,rtol=2e-5)
print('采样与训练前向的 token 对数概率一致。')

## 4 · 奖励微调：实时看模型学习

采用第三课的组内优势、策略裁剪和 KL 约束。预先固定学习率 3e-5、600 轮、组大小 8；每 60 轮显示一次曲线。测试准确率只观察，不影响更新或停止时点。

关注训练与测试两条线，以及有效组比例。如果一组回答全对或全错，就没有相对奖励信号。

In [ ]:
STEPS, GROUP, PROMPTS, UPDATES = 600, 8, 16, 2
LR, BETA, CLIP = 3e-5, 0.02, 0.2
optimizer = torch.optim.AdamW(actor.parameters(),lr=LR,weight_decay=0.0)
history = [dict(step=0,train=baseline_train,test=baseline_test,reward=None,active=None,kl=None)]
live = display('',display_id=True)
window = []
start = time.perf_counter()
for step in range(1,STEPS+1):
    length = random.choices(list(by_length),weights=[len(v) for v in by_length.values()])[0]
    pairs = random.choices(by_length[length],k=PROMPTS)
    x,mask,old_logp,rewards,answers,prompt_len = rollout(actor,pairs,group=GROUP)
    grouped = rewards.reshape(PROMPTS,GROUP)
    adv = group_advantages(grouped).reshape(-1,1)
    active_fraction = (grouped.std(-1,unbiased=False)>0).float().mean().item()
    with torch.no_grad():
        ref_logp = response_logps(reference,x,prompt_len)
    for _ in range(UPDATES):
        logp = response_logps(actor,x,prompt_len)
        ratio = (logp-old_logp).exp()
        policy = -torch.minimum(ratio*adv,ratio.clamp(1-CLIP,1+CLIP)*adv)
        delta = ref_logp-logp
        kl = delta.exp()-delta-1
        loss = (((policy+BETA*kl)*mask).sum(1)/mask.sum(1)).mean()
        assert torch.isfinite(loss), '损失出现非有限值'
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(actor.parameters(),1.0)
        optimizer.step()
    window.append((rewards.mean().item(),active_fraction,((kl.detach()*mask).sum()/mask.sum()).item()))
    if step%60==0:
        train_acc,_ = evaluate(actor,train_pairs)
        test_acc,_ = evaluate(actor,test_pairs)
        row = dict(step=step,train=train_acc,test=test_acc,
                   reward=sum(v[0] for v in window)/len(window),
                   active=sum(v[1] for v in window)/len(window),
                   kl=sum(v[2] for v in window)/len(window))
        history.append(row)
        window=[]
        fig,axes=plt.subplots(1,3,figsize=(12,3.2))
        steps=[r['step'] for r in history]
        axes[0].plot(steps,[r['train'] for r in history],label='Train')
        axes[0].plot(steps,[r['test'] for r in history],label='Test')
        axes[0].axhline(baseline_test,color='gray',ls='--',label='Frozen test baseline')
        axes[0].set(title='Greedy exact-match accuracy',ylim=(0,1.05))
        axes[0].legend(fontsize=8)
        axes[1].plot(steps[1:],[r['reward'] for r in history[1:]],label='Sample reward')
        axes[1].plot(steps[1:],[r['active'] for r in history[1:]],label='Active groups')
        axes[1].set(title='Sampling signal',ylim=(0,1.05))
        axes[1].legend(fontsize=8)
        axes[2].plot(steps[1:],[r['kl'] for r in history[1:]])
        axes[2].set(title='Sampled KL estimate')
        for ax in axes: ax.set_xlabel('Iteration')
        fig.tight_layout()
        live.update(fig)
        plt.close(fig)
        print(f"step {step:3d} | reward {row['reward']:.3f} | active {row['active']:.1%} | test {test_acc:.2%} | KL {row['kl']:.4f}",flush=True)
torch.cuda.synchronize()
elapsed=time.perf_counter()-start
print(f'训练完成，用时 {elapsed:.1f} 秒。')
rl_seconds=elapsed
rl_history=history
rl_test,rl_answers=evaluate(actor,test_pairs)
rl_train,_=evaluate(actor,train_pairs)


## 5 · 继续 SFT：比较另一种训练方式

从同一个预热起点训练另一个副本，直接用标准答案计算交叉熵。固定学习率 0.001、1200 次更新、批量 128。SFT 有更直接的 token 级监督；这个任务的标准答案可以计算，因此它是必要且强的对照。

用时分别记录。此处的实时曲线可以帮助你看到 SFT 与 RL 的不同学习过程，但不能凭一次运行推断普遍优劣。

In [ ]:
SFT_STEPS=1200
sft_optimizer=torch.optim.AdamW(sft_model.parameters(),lr=0.001,weight_decay=0.01)
sft_history=[dict(step=0,train=baseline_train,test=baseline_test)]
sft_live=display('',display_id=True)
sft_start=time.perf_counter()
for step in range(1,SFT_STEPS+1):
    sft_model.train()
    idx=torch.randint(len(train_pairs),(128,),device='cuda')
    loss=sft_model(input_ids=train_ids[idx],attention_mask=train_mask[idx],labels=train_labels[idx]).loss
    assert torch.isfinite(loss)
    sft_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(sft_model.parameters(),1.0)
    sft_optimizer.step()
    if step%120==0:
        sft_model.eval()
        tr,_=evaluate(sft_model,train_pairs)
        te,_=evaluate(sft_model,test_pairs)
        sft_history.append(dict(step=step,train=tr,test=te))
        fig,ax=plt.subplots(figsize=(7,3))
        for key in ['train','test']:
            ax.plot([r['step'] for r in sft_history],[r[key] for r in sft_history],label=key)
        ax.axhline(baseline_test,ls='--',color='gray',label='Frozen test baseline')
        ax.set(xlabel='SFT updates',ylabel='Exact-match accuracy',ylim=(0,1.05))
        ax.legend();fig.tight_layout();sft_live.update(fig);plt.close(fig)
        print(f'SFT {step:4d} | train {tr:.2%} | test {te:.2%}',flush=True)
torch.cuda.synchronize()
sft_seconds=time.perf_counter()-sft_start
sft_model.eval()
sft_test,sft_answers=evaluate(sft_model,test_pairs)
sft_train,_=evaluate(sft_model,train_pairs)


## 6 · 三组同题比较：看百分点、题数和错误转移

柱状图纵轴从 0 开始，不截断坐标放大差异。每道测试题占 1.25 个百分点。展示完整测试答案表，以及改善/退化题数；不只选择成功样本。

测试结果是域内、未在训练中出现的数对，不代表更大数字或通用推理能力。

In [ ]:
names=['Frozen baseline','Continue SFT','Reward RL']
scores=[baseline_test,sft_test,rl_test]
fig,ax=plt.subplots(figsize=(8,4))
bars=ax.bar(names,scores,color=['#94a3b8','#2563eb','#059669'])
ax.set(ylim=(0,1.12),ylabel='Held-out exact-match accuracy',title='Same 80 test problems / same initial checkpoint')
for bar,value in zip(bars,scores):
    ax.text(bar.get_x()+bar.get_width()/2,value+0.02,f'{value:.2%}',ha='center')
plt.tight_layout();plt.show()
for label,score,answers,seconds in [('SFT',sft_test,sft_answers,sft_seconds),('RL',rl_test,rl_answers,rl_seconds)]:
    improved=sum(b!=str(a+c) and f==str(a+c) for (a,c),b,f in zip(test_pairs,baseline_answers,answers))
    regressed=sum(b==str(a+c) and f!=str(a+c) for (a,c),b,f in zip(test_pairs,baseline_answers,answers))
    print(f'{label}: {score:.2%}，变化 {(score-baseline_test)*100:+.2f} 个百分点；改善 {improved} 题 / 退化 {regressed} 题；{seconds:.1f} 秒')
import pandas as pd
table=pd.DataFrame([dict(question=f'{a}+{b}=',target=str(a+b),baseline=b0,sft=s,rl=r)
    for (a,b),b0,s,r in zip(test_pairs,baseline_answers,sft_answers,rl_answers)])
display(table.style.apply(lambda row:['background-color: #dcfce7' if col in ('baseline','sft','rl') and row[col]==row['target'] else '' for col in table.columns],axis=1))
assert all(torch.equal(initial_state[k],v.detach().cpu()) for k,v in reference.state_dict().items())
assert any(not torch.equal(initial_state[k],v.detach().cpu()) for k,v in actor.state_dict().items())
print('验证通过：冻结对照没有变化，RL 模型已更新。')


## 7 · 保存证据
保存三组权重、固定划分、所有测试答案和曲线记录。重复保存会覆盖本实验结果；与前三课输出分开。

In [ ]:
out=Path('experiment04_output')
out.mkdir(exist_ok=True)
for name,m in [('baseline',reference),('sft',sft_model),('rl',actor)]:
    m.save_pretrained(out/name)
table.to_csv(out/'all_test_answers.csv',index=False)
record=dict(seed=SEED,warmup=WARMUP,train_pairs=train_pairs,test_pairs=test_pairs,vocab=vocab,
    baseline_train=baseline_train,baseline_test=baseline_test,sft_train=sft_train,sft_test=sft_test,rl_train=rl_train,rl_test=rl_test,
    rl_seconds=rl_seconds,sft_seconds=sft_seconds,rl_history=rl_history,sft_history=sft_history,
    config=dict(rl_steps=STEPS,rl_updates=UPDATES,rl_lr=LR,group=GROUP,prompts=PROMPTS,beta=BETA,clip=CLIP,
                sft_steps=SFT_STEPS,sft_lr=0.001,sft_batch=128,warmup_lr=0.002,warmup_batch=64),
    torch_version=torch.__version__,gpu=torch.cuda.get_device_name(0))
(out/'results.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
print('已保存:',out.resolve())


## 怎样解读“明显”

- 如果 RL 比冻结基线明显进步，说明这个起点上奖励更新产生了可见效果。
- 如果 SFT 更好，应报告 SFT 更好，不能宣称 RL 最优。
- 这个起点刻意只预热 200 步，是展示学习的设计条件，不是上一课已充分训练模型上的提升。
- 本实验仍然是单种子演示。要做研究结论，需要多个随机种子、验证集和未反复查看的新最终测试集。

本教程是教学版 GRPO 风格循环，未运行完整 verl 分布式训练器。